# 📊 AI Data Analyst Agent — DeepSeek R1

**How it works:**
1. Load any CSV file from your Kaggle dataset
2. Planner Agent → converts your question into a structured JSON analysis plan
3. Data Worker → executes the plan using pandas
4. Chart Generator → builds a chart with matplotlib/seaborn
5. Explainer Agent → summarizes results in plain English

**Setup required before running:**
- Go to **Add-ons → Secrets** in Kaggle and add your key:
  - Key name: `DEEPSEEK_API_KEY`
  - Value: your API key from https://platform.deepseek.com
- Add your CSV dataset to this notebook via **Add Data**

## Cell 1 — Install Required Packages

In [ ]:
# Install the OpenAI-compatible client for DeepSeek API
# All other libraries (pandas, matplotlib, seaborn) are pre-installed in Kaggle
!pip install openai --quiet

## Cell 2 — Imports

In [ ]:
# Standard library imports
import io           # Holds chart image in memory before displaying
import json         # Parse and create JSON plans
import re           # Regex to extract JSON from model responses
from datetime import datetime  # For timestamping output files

# Data and visualization
import matplotlib.pyplot as plt   # Draw charts
import matplotlib.image as mpimg  # Display charts inline in notebook
import pandas as pd               # Load CSV and run data operations
import seaborn as sns             # Better chart colors and styles

# Kaggle Secrets — safe way to read API keys without hardcoding
from kaggle_secrets import UserSecretsClient

# DeepSeek uses OpenAI-compatible API format
from openai import OpenAI

# IPython display utilities for rich notebook output
from IPython.display import display, Markdown, JSON

# Set chart style globally
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All imports successful.")

## Cell 3 — DeepSeek Client Setup

> **Important:** Before running this cell, add your API key in Kaggle:
> `Add-ons` → `Secrets` → Add `DEEPSEEK_API_KEY`

In [ ]:
# Read API key securely from Kaggle Secrets
# This avoids hardcoding the key in your notebook (which would be public!)
secrets = UserSecretsClient()
DEEPSEEK_API_KEY = secrets.get_secret("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise RuntimeError(
        "DEEPSEEK_API_KEY not found.\n"
        "Go to Add-ons → Secrets → Add key named 'DEEPSEEK_API_KEY'"
    )

# DeepSeek's base URL — they use the same interface as OpenAI
DEEPSEEK_BASE_URL = "https://api.deepseek.com"

# Model name: deepseek-reasoner = DeepSeek R1
# This model thinks step-by-step before answering (chain-of-thought style)
MODEL_NAME = "deepseek-reasoner"

# Create the client object — all LLM calls go through this
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

print(f"✅ DeepSeek client ready. Model: {MODEL_NAME}")

## Cell 4 — Helper Functions
Utility functions for schema description, date preprocessing, filters, and charting.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# HELPER 1: get_schema_description
# Converts a DataFrame into a text description the Planner Agent
# can read. The model cannot see the DataFrame directly — it needs
# a text summary to pick valid column names and understand data types.
# ─────────────────────────────────────────────────────────────────

def get_schema_description(df: pd.DataFrame, max_rows: int = 3) -> str:
    """
    Describe a DataFrame as plain text for the LLM.
    
    Example output:
        Columns:
          - Category (object) — sample: Technology, Furniture
          - Sales (float64)
        Sample rows:
        | Category   | Sales |
        | Technology | 1200  |
    """
    lines = ["Columns:"]

    for col, dtype in df.dtypes.items():
        # For low-cardinality text columns, show sample values
        # so the model knows what values are possible (e.g., category names)
        if dtype == "object" and df[col].nunique() < 20:
            samples = df[col].dropna().unique()[:5]
            lines.append(f"  - {col} ({dtype}) — sample: {', '.join(map(str, samples))}")
        else:
            lines.append(f"  - {col} ({dtype})")

    # Append first few rows as a markdown table for more context
    sample_table = df.head(max_rows).to_markdown(index=False)
    return "\n".join(lines) + "\n\nSample rows:\n" + sample_table


# ─────────────────────────────────────────────────────────────────
# HELPER 2: preprocess_dates
# Automatically detects date columns and extracts year/month features.
# This lets users ask questions like "sales last year" or "by month"
# without manually engineering time features.
# ─────────────────────────────────────────────────────────────────

def preprocess_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect date columns and create helper columns:
      - <col>_year        → integer year  (e.g. 2023)
      - <col>_month       → integer month (e.g. 7)
      - <col>_year_month  → string period (e.g. '2023-07')
    """
    df = df.copy()
    date_columns = []

    for col in df.columns:
        # Already a datetime type — add to list immediately
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            date_columns.append(col)
            continue

        # Column name hints at a date (order_date, ship_time, day, etc.)
        if any(kw in col.lower() for kw in ["date", "time", "day"]):
            try:
                parsed = pd.to_datetime(df[col], errors="coerce")
                # Only keep if at least some rows parsed successfully
                if parsed.notna().sum() > 0:
                    df[col] = parsed
                    date_columns.append(col)
            except Exception:
                pass  # Not a real date column — skip silently

    # Create year / month / year_month features for each date column
    for col in date_columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[f"{col}_year"]       = df[col].dt.year
            df[f"{col}_month"]      = df[col].dt.month
            df[f"{col}_year_month"] = df[col].dt.to_period("M").astype(str)

    return df


# ─────────────────────────────────────────────────────────────────
# HELPER 3: apply_filters
# Applies row filters from the planner's JSON plan to the DataFrame.
# Example filter: {"column": "Year", "op": "==", "value": 2023}
# ─────────────────────────────────────────────────────────────────

def apply_filters(df: pd.DataFrame, filters: list) -> pd.DataFrame:
    """
    Apply a list of filter dicts to a DataFrame.
    Supported operators: ==, !=, >, <, >=, <=, contains
    Filters are applied sequentially (AND logic).
    """
    if not filters:
        return df

    filtered = df.copy()

    for f in filters:
        col = f.get("column")
        op  = f.get("op")
        val = f.get("value")

        # Skip filters that reference a non-existent column
        if col not in filtered.columns:
            print(f"⚠️  Filter column '{col}' not found — skipping.")
            continue

        # Try to cast value to a number for numeric comparisons
        try:
            val_cast = pd.to_numeric(val)
        except Exception:
            val_cast = val  # Keep as string if it can't be numeric

        # Apply the appropriate comparison
        if   op == "==":      filtered = filtered[filtered[col] == val_cast]
        elif op == "!=":      filtered = filtered[filtered[col] != val_cast]
        elif op == ">":       filtered = filtered[filtered[col] >  val_cast]
        elif op == "<":       filtered = filtered[filtered[col] <  val_cast]
        elif op == ">=":      filtered = filtered[filtered[col] >= val_cast]
        elif op == "<=":      filtered = filtered[filtered[col] <= val_cast]
        elif op == "contains":
            # Case-insensitive substring match
            filtered = filtered[
                filtered[col].astype(str).str.contains(
                    str(val_cast), case=False, na=False
                )
            ]

    return filtered


# ─────────────────────────────────────────────────────────────────
# HELPER 4: run_analysis_plan
# The "Data Worker" — reads the JSON plan and runs pandas operations.
# The LLM does NOT write or run code; it only decides WHAT to compute.
# This function decides HOW to compute it safely.
# ─────────────────────────────────────────────────────────────────

def run_analysis_plan(df: pd.DataFrame, plan: dict) -> pd.DataFrame:
    """
    Execute the JSON analysis plan using pandas.

    Supported operations: group_by_summary, aggregate_compare, filter_then_aggregate
    Supported metrics:    sum, mean, count, max, min
    """
    op         = plan.get("operation")
    target_col = plan.get("target_column")
    group_by   = plan.get("group_by") or []
    filters    = plan.get("filters") or []
    metric     = plan.get("metric", "sum")

    # Step 1: Apply row filters first
    if filters:
        df = apply_filters(df, filters)

    # Step 2: Validate the target column
    if not op or not target_col or target_col not in df.columns:
        print(f"⚠️  Column '{target_col}' not found. Showing raw data preview.")
        return df.head(20)

    # Step 3: Run the aggregation
    SUPPORTED_OPS = ("group_by_summary", "aggregate_compare", "filter_then_aggregate")
    VALID_METRICS = ("sum", "mean", "count", "max", "min")

    if op in SUPPORTED_OPS:

        if metric not in VALID_METRICS:
            raise ValueError(f"Unsupported metric: '{metric}'. Use one of: {VALID_METRICS}")

        if not group_by:
            # No group by — aggregate the whole column into a single number
            agg_func = getattr(df[target_col], metric)
            value    = agg_func()
            return pd.DataFrame({f"{metric}_{target_col}": [value]})

        # Group by one or more columns, then aggregate
        agg_df = (
            df.groupby(group_by)[target_col]
            .agg(metric)
            .reset_index()
        )

        # Sort descending so the top results appear first
        agg_df = agg_df.sort_values(by=target_col, ascending=False)
        return agg_df

    # Unknown operation — fall back to a preview
    return df.head(20)


# ─────────────────────────────────────────────────────────────────
# HELPER 5: generate_chart
# Builds a bar, line, or pie chart from the result DataFrame.
# Returns None if the plan doesn't need a chart or data is empty.
# ─────────────────────────────────────────────────────────────────

def generate_chart(df: pd.DataFrame, plan: dict):
    """
    Draw a chart based on the plan's chart_type (bar / line / pie).
    Displays inline in the Kaggle notebook using plt.show().
    Also returns a BytesIO PNG buffer for saving if needed.
    """
    # Skip chart if not requested or no data
    if not plan.get("need_chart") or df.empty:
        return None

    chart_type = plan.get("chart_type", "bar")
    group_by   = plan.get("group_by") or []
    target_col = plan.get("target_column")

    if not target_col or target_col not in df.columns:
        return None

    fig, ax = plt.subplots(figsize=(12, 6))

    if group_by:
        x_col   = group_by[0]              # First group-by column becomes the X axis
        plot_df = df.head(15)              # Cap at 15 rows to keep chart readable

        if x_col not in df.columns:
            plt.close(fig)
            return None

        if chart_type == "bar":
            # Bar chart — good for comparing categories
            colors = sns.color_palette("husl", len(plot_df))
            bars   = ax.bar(range(len(plot_df)), plot_df[target_col], color=colors)
            ax.set_xticks(range(len(plot_df)))
            ax.set_xticklabels(plot_df[x_col], rotation=45, ha="right", fontsize=10)
            ax.set_ylabel(target_col, fontsize=12)
            ax.set_xlabel(x_col, fontsize=12)

            # Add value labels on top of each bar
            for bar, val in zip(bars, plot_df[target_col]):
                label = f"{val:,.0f}" if abs(val) > 100 else f"{val:.2f}"
                ax.text(
                    bar.get_x() + bar.get_width() / 2.0,
                    bar.get_height(),
                    label,
                    ha="center", va="bottom", fontsize=9
                )

        elif chart_type == "line":
            # Line chart — good for time series / trends
            ax.plot(
                range(len(plot_df)), plot_df[target_col],
                marker="o", linewidth=2, markersize=8, color="#1f77b4"
            )
            ax.set_xticks(range(len(plot_df)))
            ax.set_xticklabels(plot_df[x_col], rotation=45, ha="right", fontsize=10)
            ax.set_ylabel(target_col, fontsize=12)
            ax.set_xlabel(x_col, fontsize=12)
            ax.grid(True, alpha=0.3)

        elif chart_type == "pie" and len(plot_df) <= 10:
            # Pie chart — good for proportions with few categories
            ax.pie(
                plot_df[target_col],
                labels=plot_df[x_col],
                autopct="%1.1f%%",
                startangle=90
            )
            ax.axis("equal")  # Keep the circle round

        else:
            # Fallback: simple bar chart
            ax.bar(range(len(plot_df)), plot_df[target_col])
            ax.set_xticks(range(len(plot_df)))
            ax.set_xticklabels(plot_df[x_col], rotation=45, ha="right")

        chart_title = f"{target_col} by {x_col}"

    else:
        # No group-by — show a single value bar
        ax.bar([target_col], [df[target_col].iloc[0]], color="#1f77b4")
        chart_title = target_col

    ax.set_title(chart_title, fontsize=14, fontweight="bold", pad=20)
    fig.tight_layout()

    # Display chart inline inside the notebook
    plt.show()

    # Also save to a BytesIO buffer (useful if you want to export later)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    plt.close(fig)  # Free memory

    return buf


print("✅ All helper functions defined.")

## Cell 5 — Planner Agent
Converts a natural language question into a structured JSON analysis plan.

In [ ]:
def call_planner_llm(schema_text: str, question: str) -> tuple[dict, str]:
    """
    Send the dataset schema + user question to DeepSeek.
    The model returns a JSON plan describing what analysis to run.

    Returns:
        plan (dict)              — structured analysis plan
        reasoning_content (str) — model's internal thinking trace (for debugging)
    """

    # System prompt defines the model's role and expected output format
    system_prompt = """
    You are a data analysis planner.
    Given a dataset schema and a user question, output a JSON analysis plan.

    Required JSON format:
    {
      "operation": "group_by_summary",
      "group_by": ["column_name"],
      "filters": [],
      "target_column": "column_to_analyze",
      "metric": "sum",
      "need_chart": true,
      "chart_type": "bar"
    }

    Field rules:
    - operation    : Always use "group_by_summary"
    - group_by     : List of columns to group by, e.g. ["Category"] or ["Region", "Year"]
    - filters      : Optional row filters, e.g. [{"column": "Year", "op": "==", "value": 2023}]
    - target_column: Column to aggregate — MUST exist in the schema
    - metric       : One of: sum, mean, count, max, min
    - need_chart   : Always true
    - chart_type   : One of: bar, line, pie

    Rules:
    - Only use columns that exist in the schema
    - For "this year" or "last year", look for year-related columns
    - For "top N" questions, use group_by with the appropriate metric
    - Output the complete JSON at the end
    """.strip()

    # Build the user message with schema context and the actual question
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"Dataset Schema:\n{schema_text}\n\n"
                f"Question:\n{question}\n\n"
                f"Now output the complete JSON plan."
            ),
        },
    ]

    # Call DeepSeek R1 — reasoning models need higher token limits
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=8192,
    )

    message          = resp.choices[0].message
    content          = (message.content or "").strip()
    finish_reason    = resp.choices[0].finish_reason

    # reasoning_content is where DeepSeek R1 stores its chain-of-thought
    # The final answer (JSON) is often embedded here
    reasoning_content = getattr(message, "reasoning_content", None) or ""

    # ── JSON Extraction ─────────────────────────────────────────────
    # DeepSeek R1 doesn't always return clean JSON — it may wrap it
    # in markdown, add prose before/after, or put it in reasoning_content.
    # We try three strategies to extract a valid plan dict.

    def extract_json_from_text(text: str) -> dict | None:
        """Try to extract a valid JSON analysis plan from a text string."""
        if not text:
            return None

        # Strategy 1: The entire text is already valid JSON
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Strategy 2: JSON is wrapped in a markdown code block ```json ... ```
        code_block = re.search(r"```(?:json)?\s*(\{[\s\S]*?\})\s*```", text, re.DOTALL)
        if code_block:
            try:
                return json.loads(code_block.group(1))
            except json.JSONDecodeError:
                pass

        # Strategy 3: Scan text for balanced { } blocks and parse each one
        def find_json_objects(s: str) -> list[str]:
            """Find all top-level JSON objects in a string via brace matching."""
            objects, depth, start = [], 0, None
            in_string, escape      = False, False
            for i, ch in enumerate(s):
                if escape:
                    escape = False
                    continue
                if ch == "\\":
                    escape = True
                    continue
                if ch == '"':
                    in_string = not in_string
                elif not in_string:
                    if ch == "{":
                        if depth == 0:
                            start = i
                        depth += 1
                    elif ch == "}":
                        depth -= 1
                        if depth == 0 and start is not None:
                            objects.append(s[start: i + 1])
                            start = None
            return objects

        # Only accept objects that look like an analysis plan
        plan_keys = {"operation", "target_column", "metric", "group_by"}
        for candidate in find_json_objects(text):
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, dict) and plan_keys & parsed.keys():
                    return parsed
            except json.JSONDecodeError:
                continue

        return None  # All strategies failed

    # Try reasoning_content first — R1 often puts the JSON there
    if reasoning_content:
        plan = extract_json_from_text(reasoning_content)
        if plan is not None:
            return plan, reasoning_content

    # Then try the main content field
    plan = extract_json_from_text(content)
    if plan is not None:
        return plan, reasoning_content

    # Both failed — raise a descriptive error
    err = (
        f"Planner could not produce valid JSON.\n"
        f"Finish reason: {finish_reason}\n"
        f"Content sample: {repr(content[:300])}\n"
    )
    if finish_reason == "length":
        err += "⚠️  Token limit reached — JSON was cut off. Try a shorter question."
    raise ValueError(err)


print("✅ Planner Agent defined.")

## Cell 6 — Explainer Agent
Turns raw analysis results into a plain-English insight summary.

In [ ]:
def call_explainer_llm(question: str, plan: dict, result_summary: list) -> tuple[str, str]:
    """
    Send the analysis result to DeepSeek and ask for a business-friendly summary.

    Args:
        question       — the original user question
        plan           — the JSON plan that was executed
        result_summary — top rows of the result DataFrame as a list of dicts

    Returns:
        explanation (str)        — plain-English insights
        reasoning_content (str) — model's thinking trace
    """

    # Tell the model to act as a business analyst, not a technician
    system_prompt = """
    You are a senior data analyst explaining results to business stakeholders.
    You will receive a question, an analysis plan, and the results.

    Your response must:
    - Start with a direct 1-2 sentence answer to the question
    - Include 3-5 bullet points highlighting key insights, trends, or comparisons
    - Use specific numbers and percentages from the data
    - Avoid technical jargon (no "aggregation", "groupby", "DataFrame")
    - End with a brief recommendation or suggested next step if relevant

    Keep it concise, factual, and actionable.
    """.strip()

    # Limit result rows sent to the model to avoid wasting tokens
    limited_summary = result_summary[:20]

    user_content = (
        f"User Question:\n{question}\n\n"
        f"Analysis Plan:\n{json.dumps(plan, indent=2)}\n\n"
        f"Results (top rows):\n{json.dumps(limited_summary, indent=2)}\n\n"
        f"Total rows in result: {len(result_summary)}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]

    # Explainer needs fewer tokens than the planner
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=2048,
    )

    message           = resp.choices[0].message
    content           = (message.content or "").strip()
    reasoning_content = getattr(message, "reasoning_content", None) or ""

    # Fallback: if main content is empty, use reasoning content
    if not content and reasoning_content:
        content = reasoning_content.strip()

    return (
        content or "⚠️  Could not generate an explanation.",
        reasoning_content,
    )


print("✅ Explainer Agent defined.")

## Cell 7 — Main `analyze()` Function
Combines everything into one function call.

In [ ]:
def analyze(df: pd.DataFrame, question: str, show_reasoning: bool = False) -> pd.DataFrame:
    """
    Run the full analysis pipeline on a DataFrame.

    Steps:
        1. Build schema description for the LLM
        2. Call Planner Agent → get JSON plan
        3. Run Data Worker   → execute plan with pandas
        4. Generate Chart    → display inline in notebook
        5. Call Explainer    → print plain-English insights

    Args:
        df              — the loaded DataFrame (already preprocessed)
        question        — your natural language question about the data
        show_reasoning  — set True to print the model's thinking trace

    Returns:
        result_df (pd.DataFrame) — the aggregated result table
    """

    print("=" * 60)
    print(f"❓ Question: {question}")
    print("=" * 60)

    # ── Step 1: Build schema description ──────────────────────────
    schema_text = get_schema_description(df)

    # ── Step 2: Planner Agent — generate JSON plan ─────────────────
    print("\n🤔 DeepSeek is planning the analysis...")
    try:
        plan, planner_reasoning = call_planner_llm(schema_text, question)
    except Exception as e:
        print(f"❌ Planner failed: {e}")
        return pd.DataFrame()

    # Display the JSON plan so you can see what the model decided to do
    print("\n📋 Analysis Plan:")
    display(JSON(plan))

    # Optionally show the model's chain-of-thought reasoning
    if show_reasoning and planner_reasoning:
        print("\n🧠 Planner Reasoning (first 800 chars):")
        print(planner_reasoning[:800] + ("..." if len(planner_reasoning) > 800 else ""))

    # ── Step 3: Data Worker — execute the plan ──────────────────────
    print("\n⚙️  Running analysis with pandas...")
    try:
        result_df = run_analysis_plan(df, plan)
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        print("💡 Tip: Try rephrasing your question or mention a specific column name.")
        return pd.DataFrame()

    # ── Step 4: Chart Generator — draw and display inline ───────────
    print("\n📊 Chart:")
    generate_chart(result_df, plan)  # plt.show() is called inside this function

    # Show key statistics for the target column
    target_col = plan.get("target_column")
    if target_col and target_col in result_df.columns:
        print("\n📈 Key Metrics:")
        col_data = result_df[target_col]
        print(f"   Total   : {col_data.sum():,.2f}")
        print(f"   Average : {col_data.mean():,.2f}")
        print(f"   Max     : {col_data.max():,.2f}")
        print(f"   Min     : {col_data.min():,.2f}")
        print(f"   Rows    : {len(result_df)}")

    # Display the result table
    print("\n📋 Results Table:")
    display(result_df)

    # ── Step 5: Explainer Agent — generate insights ─────────────────
    print("\n💡 Generating AI insights...")

    # Convert datetime/period columns to strings before JSON serialization
    result_serializable = result_df.copy()
    for col in result_serializable.columns:
        if pd.api.types.is_datetime64_any_dtype(result_serializable[col]):
            result_serializable[col] = result_serializable[col].astype(str)
        else:
            # Handle Period dtype (e.g. year_month columns)
            try:
                result_serializable[col] = result_serializable[col].astype(str)
            except Exception:
                pass

    result_summary = result_serializable.to_dict(orient="records")

    try:
        explanation, explainer_reasoning = call_explainer_llm(
            question=question,
            plan=plan,
            result_summary=result_summary,
        )
    except Exception as e:
        explanation = f"❌ Explainer failed: {e}"
        explainer_reasoning = ""

    # Print the final insight summary
    print("\n" + "=" * 60)
    print("💡 AI INSIGHTS")
    print("=" * 60)
    display(Markdown(explanation))

    # Optionally show the explainer's reasoning trace
    if show_reasoning and explainer_reasoning:
        print("\n🧠 Explainer Reasoning (first 800 chars):")
        print(explainer_reasoning[:800] + ("..." if len(explainer_reasoning) > 800 else ""))

    print("\n" + "=" * 60)

    return result_df


print("✅ analyze() function ready.")

## Cell 8 — Load Your CSV

Change the path below to match your Kaggle dataset file.
All Kaggle datasets are available at `/kaggle/input/<dataset-name>/`

In [ ]:
import os

# ── CHANGE THIS PATH to your own dataset ──────────────────────────
# Example paths:
#   /kaggle/input/superstore-sales/superstore.csv
#   /kaggle/input/hr-analytics/HR_Analytics.csv
CSV_PATH = "/kaggle/input/your-dataset/your-file.csv"
# ──────────────────────────────────────────────────────────────────

# Auto-find the CSV if you're not sure of the exact path
if not os.path.exists(CSV_PATH):
    print("⚠️  CSV path not found. Searching /kaggle/input/ ...")
    for root, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".csv"):
                found_path = os.path.join(root, file)
                print(f"   Found: {found_path}")
    print("\n👆 Copy one of the paths above and set it as CSV_PATH.")
else:
    # Load the CSV and run date preprocessing
    df = pd.read_csv(CSV_PATH)
    df = preprocess_dates(df)  # Auto-creates year/month columns from date fields

    # Show a quick overview
    print(f"✅ Loaded: {CSV_PATH}")
    print(f"   Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Columns: {list(df.columns)}")
    print()
    display(df.head())

## Cell 9 — Run Your Analysis

Change the `QUESTION` variable to anything you want to ask about your data.
Set `show_reasoning=True` to see the model's internal thinking.

In [ ]:
# ── CHANGE THIS to your question ──────────────────────────────────
QUESTION = "Which product category had the highest total sales?"
# ──────────────────────────────────────────────────────────────────

# Run the full pipeline
# show_reasoning=True prints the model's chain-of-thought (useful for debugging)
result = analyze(df, question=QUESTION, show_reasoning=False)

## Cell 10 — Ask More Questions (Optional)

You can run `analyze()` as many times as you want with different questions.

In [ ]:
# Example: ask a different question on the same dataset
result2 = analyze(df, question="Show monthly sales trend for the last year")

In [ ]:
# Example: ask about a specific region or segment
result3 = analyze(df, question="Which region has the highest average profit?")

---
## 💾 Save Results (Optional)
Save any result DataFrame to a CSV file in the Kaggle working directory.

In [ ]:
# Save the result from Cell 9 as a CSV file
output_path = "/kaggle/working/analysis_result.csv"
result.to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")
print("   You can download this from the Output tab on the right.")